In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import gym
from torch.distributions import MultivariateNormal

In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim))
        
    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = torch.relu(self.fc2(x))
        mean = self.fc3(x)
        std = torch.exp(self.log_std)
        return mean, std

In [ ]:
class ValueNetwork(nn.Module):
    def __init__(self, state_dim):
        super(ValueNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 1)
        
    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

In [ ]:
class TRPOAgent:
    def __init__(self, state_dim, action_dim):
        self.policy = PolicyNetwork(state_dim, action_dim)
        self.value_function = ValueNetwork(state_dim)
        self.value_optimizer = optim.Adam(self.value_function.parameters(), lr=1e-3)
    
    def select_action(self, state):
        state = torch.tensor(state, dtype=torch.float32)
        mean, std = self.policy(state)
        dist = MultivariateNormal(mean, torch.diag(std))
        action = dist.sample()
        return action.detach().numpy(), dist.log_prob(action).detach()
    
    def compute_advantage(self, rewards, values):
        advantages = rewards - values.detach().numpy()
        return torch.tensor(advantages, dtype=torch.float32)
    
    def conjugate_gradient(self, fisher_vector_product, b, iter=10, residual_tol=1e-10):
        x = torch.zeros_like(b)
        r = b.clone()
        p = b.clone()
        rdotr = torch.dot(r, r)
        
        for _ in range(iter):
            Ap = fisher_vector_product(p)
            alpha = rdotr / torch.dot(p, Ap)
            x += alpha * p
            r -= alpha * Ap
            new_rdotr = torch.dot(r, r)
            if new_rdotr < residual_tol:
                break
            beta = new_rdotr / rdotr
            p = r + beta * p
            rdotr = new_rdotr
        
        return x
    
    def fisher_vector_product(self, v):
        mean, std = self.policy(torch.zeros_like(v))
        dist = MultivariateNormal(mean, torch.diag(std))
        log_probs = dist.log_prob(torch.zeros_like(v))
        kl = torch.mean(log_probs)
        grads = torch.autograd.grad(kl, self.policy.parameters(), create_graph=True)
        flat_grads = torch.cat([grad.view(-1) for grad in grads])
        kl_v = torch.dot(flat_grads.t(), v)
        grads_2 = torch.autograd.grad(kl_v, self.policy.parameters())
        flat_grads_2 = torch.cat([grad.view(-1) for grad in grads_2])
        return flat_grads_2 + 0.1 * v
    
    def update_policy(self, states, actions, log_probs_old, advantages):
        # select_action(states)
        mean, std = self.policy(states)
        dist = MultivariateNormal(mean, torch.diag(std))
        log_probs = dist.log_prob(actions)
        # ===========================================
        
        ent = dist.entropy()
        ratio = torch.exp(log_probs - log_probs_old)
        surrogate_loss = -torch.mean(ratio * advantages)-ent
        grads = torch.autograd.grad(surrogate_loss, self.policy.parameters())
        j = torch.cat([g.view(-1) for g in grads])
        
        def categorical_kl(p, q):
            ratio = p / (q + 1e-6)
            ratio[p==0] = 1
            ratio[(q==0) & (p!=0)] = torch.inf
            return (p * torch.log(ratio)).sum(dim=1)


        #kl_divergence = torch.mean(categorical_kl(log_probs, log_probs_old))
        
        def fisher_vector_product(v):
            kl = torch.mean(categorical_kl(log_probs, log_probs_old))
            grads = torch.autograd.grad(kl, self.policy.parameters(), create_graph=True)
            flat_grads = torch.cat([grad.view(-1) for grad in grads])
            kl_v = torch.dot(flat_grads.t(), v)
            grads_2 = torch.autograd.grad(kl_v, self.policy.parameters())
            flat_grads_2 = torch.cat([grad.view(-1) for grad in grads_2])
            return flat_grads_2 + 0.1 * v
        
        search_direction = self.conjugate_gradient(fisher_vector_product, -advantages)
        max_step_size = torch.sqrt(2 * 0.01 / (torch.dot(search_direction, fisher_vector_product(search_direction)) + 1e-8))
        step = max_step_size * search_direction
        
        with torch.no_grad():
            for param, step_param in zip(self.policy.parameters(), step):
                param.add_(step_param)